## Plot Graphs

In [ ]:
import polars as pl
import pandas as pd
import networkx as nx

import matplotlib.pyplot as plt
reviewers_projection = pl.read_ipc("../reviewers projection/reviewers_graph.feather")
revisions = pl.read_parquet("../revisions_no_bots.parquet")

ModuleNotFoundError: No module named 'networkx'

## Reviewers

In [9]:
revisions.head()

revid,parentid,minor,user,userid,timestamp,size,slots,tags,pageid,title
i64,i64,bool,str,f64,str,i64,struct[1],list[str],i64,str
1163683705,1073278143,false,"""Smasongarrison""",1.6185737e7,"""2023-07-06T02:12:52Z""",14787,"{{""wikitext"",14787}}","[""AWB""]",57185536,"""Georgia Hopley"""
1170257280,1169998062,true,"""Smasongarrison""",1.6185737e7,"""2023-08-14T00:52:02Z""",39439,"{{""wikitext"",39439}}",[],15394015,"""Willis Ward"""
1162127828,1160202917,true,"""Minorax""",3.6529075e7,"""2023-06-27T04:21:11Z""",39390,"{{""wikitext"",39390}}","[""wikieditor""]",15394015,"""Willis Ward"""
1160202917,1160202005,false,"""SheridanFord""",4.263828e6,"""2023-06-15T01:20:50Z""",39421,"{{""wikitext"",39421}}","[""visualeditor""]",15394015,"""Willis Ward"""
1160202005,1152748211,false,"""SheridanFord""",4.263828e6,"""2023-06-15T01:11:16Z""",38839,"{{""wikitext"",38839}}","[""visualeditor""]",15394015,"""Willis Ward"""


In [10]:
r = (
    reviewers_projection.with_columns([
        pl.min_horizontal("from", "to").alias("source_sorted"),
        pl.max_horizontal("from", "to").alias("target_sorted")
    ])
    .unique(subset=["source_sorted", "target_sorted"])
    .select(["from", "to"])
    .join(
        revisions.select(["userid", "user"]).unique(),
        left_on="from",
        right_on="userid",
        how="left"
    )
    .rename({"user": "from_user"})
    .join(
        revisions.select(["userid", "user"]).unique(),
        left_on="to",
        right_on="userid",
        how="left"
    )
    .rename({"user": "to_user"})  
)

In [12]:
G = nx.from_pandas_edgelist(r.to_pandas(), source='from_user', target='to_user')

# Calculate degrees (number of connections) for all nodes
degrees = dict(G.degree())

# Sort nodes by degree in descending order and pick the top 5
top_n = 5
sorted_degrees = sorted(degrees.items(), key=lambda item: item[1], reverse=True)
top_nodes = [node for node, degree in sorted_degrees[:top_n]]

# Create a label dictionary containing ONLY the top nodes
labels = {node: node for node in top_nodes}

# Initialize plot
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.15, seed=42)

# Draw the graph components
nx.draw_networkx_nodes(G, pos, node_size=50, node_color='#3a81ba')
nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='gray')

# Draw labels only for the top nodes
# We use a larger font and bold weight to make them stand out
nx.draw_networkx_labels(G, pos, labels, font_size=12, font_color='black', font_weight='bold')

plt.title(f"Graph Fragment with Top {top_n} Most Connected Nodes Labeled")
plt.axis('off')
plt.show()

NameError: name 'nx' is not defined